# Lecția 12 - Reducerea Istoricului de Chat cu Agent Scratchpad

Acest notebook demonstrează cum să gestionăm contextul în conversații lungi folosind Microsoft Agent Framework. Pe măsură ce conversațiile cresc, numărul de tokeni crește — în cele din urmă depășind fereastra de context a modelului. Abordăm acest lucru printr-un **pattern de sumarizare a contextului** și un **agent scratchpad** pentru memorie persistentă.

## Ce Vei Învăța:
1. **De ce Contează Gestionarea Contextului**: Înțelegerea limitelor de tokeni și a ferestrelor de context
2. **Agenti Conștienți de Context**: Construirea agenților care gestionează propriul context de conversație
3. **Pattern-ul de Sumarizare a Contextului**: Utilizarea uneltelor pentru a condensa istoricul conversației
4. **Agent Scratchpad**: Memorie persistentă care supraviețuiește reducerii contextului

## Cerințe Prealabile:
- Configurarea Azure OpenAI cu variabilele de mediu configurate
- Înțelegerea conceptelor de bază despre agenți din lecțiile anterioare


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## De ce contează gestionarea contextului

Fiecare LLM are o **fereastră de context** finită — numărul maxim de tokeni pe care îi poate procesa într-o singură cerere. Pe măsură ce o conversație multi-turn evoluează:

- **Numărul de tokeni crește liniar** cu fiecare mesaj al utilizatorului și răspunsul asistentului.
- **Tokenii din prompt domină costul** deoarece întreaga istorie este retrimisă la fiecare rundă.
- În cele din urmă conversația **depășește fereastra de context** și modelul ori trunchiază, ori returnează o eroare.

### Strategii pentru gestionarea contextului

| Strategie | Cum funcționează | Compromis |
|---|---|---|
| **Trunchiere** | Se elimină cele mai vechi mesaje | Se pierde contextul inițial |
| **Rezumat** | Se condensează mesajele mai vechi într-un rezumat | Se pierd unele detalii, dar punctele cheie sunt păstrate |
| **Scratchpad / Memorie externă** | Se stochează informații cheie în afara conversației | Necesită apeluri către unelte, dar supraviețuiește oricărei reduceri |

În acest notebook combinăm **rezumatul** cu un **instrument scratchpad** astfel încât agentul să poată menține continuitatea chiar și când istoricul conversației este condensat.


## Crearea unui Agent Conștient de Context


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simularea unei conversații lungi

Să parcurgem o conversație pe mai multe runde pentru a vedea cum se acumulează contextul. Agentul ar trebui să rețină detalii cheie (preferințe, buget, datele călătoriei) pe parcursul rundelor și să demonstreze continuitate.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Observă cum agentul păstrează contextul de la intervențiile anterioare — știe despre Japonia, sushi, temple, fotografie, bugetul de 3000 de dolari, călătoria solo și perioada din aprilie. Într-o conversație scurtă, acest lucru funcționează bine, dar pe măsură ce conversația crește, întreaga istorie devine costisitor de retrimis.

Să continuăm conversația cu mai multe intervenții pentru a vedea acumularea contextului:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Model de rezumare a contextului

Pe măsură ce conversația crește, putem folosi un **instrument de rezumare** pentru a comprima contextul acumulat într-un format compact. Agentul apelează acest instrument pentru a înregistra preferințele cheie astfel încât, chiar dacă mesajele mai vechi sunt eliminate, informațiile esențiale să fie păstrate.

Acest model este piatra de temelie pentru o reducere mai sofisticată a istoricului:
1. Agentul identifică faptele cheie din conversație
2. Apelează instrumentul de rezumare pentru a le păstra
3. Mesajele mai vechi pot fi eliminate în siguranță deoarece rezumatul surprinde ceea ce contează

Mai jos definim un instrument `summarize_preferences` pe care agentul îl poate apela pentru a înregistra un rezumat compact al ceea ce a învățat.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Rezumat

În această lecție ai învățat cum să gestionezi contextul în conversațiile agenților care durează mult folosind Microsoft Agent Framework:

### Concepte cheie
- **Ferestrele de context sunt finite** — fiecare token din istoricul conversației costă bani și se contabilizează către limită.
- **Instrumentele de sumarizare** permit agentului să condenseze contextul acumulat în rezumate compacte, reducând utilizarea tokenilor în timp ce păstrează informațiile esențiale.
- **Blocnotesurile agenților** oferă o memorie externă persistentă care supraviețuiește oricărei reduceri a conversației.

### Ce ai construit
- Un **agent conștient de context** care menține continuitatea în conversații cu mai multe schimburi
- Un **instrument de sumarizare** (`summarize_preferences`) care înregistrează detalii cheie despre utilizator într-un format compact
- O **conversație cu mai multe schimburi** care demonstrează păstrarea contextului și gestionarea schimbărilor

### Aplicații în lumea reală
- **Boti de Servicii Clienți**: Păstrează preferințele pe durata unor sesiuni lungi de suport
- **Asistenți Personali**: Urmăresc proiectele în desfășurare fără a fi nevoie de reexprimarea contextului
- **Tutorii Educaționali**: Mențin progresul elevului pe parcursul multor interacțiuni

### Pașii următori
- Implementarea unui instrument complet de blocnotes cu persistență bazată pe fișiere
- Adăugarea truncării automate a istoricului după sumarizare
- Combinarea cu baze de date vectoriale pentru căutarea memoriei semantice
- Construirea de agenți care pot relua conversațiile zile mai târziu cu context complet


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere automată AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa autoritară. Pentru informații critice, se recomandă traducerea profesională realizată de un specialist uman. Nu ne asumăm răspunderea pentru eventuale neînțelegeri sau interpretări greșite care pot apărea ca urmare a utilizării acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
